# Data Storytelling with CBP Border Crossing Entry Data

In [234]:
# In this project I used data files from the following links:
#https://data.gov/
#https://www.cbp.gov/newsroom/stats/cbp-public-data-portal

library(ggplot2)
library(dplyr)
library(tidyr)

#Read the csv file to dataframe
border_data = read.csv('data/Border_Crossing_Entry_Data.csv')
#head(border_data)


#Some data carpentry for Date column
#seperate Year and Month in order to tidy the data
border_data<-border_data %>%
                    separate(Date, into = c("Month", "Year"), sep = " ")

#Convert Year attribute to integer data type.
border_data <- border_data %>%
          mutate(Year = as.integer(Year))

#There is one row with NA values.I will remove it.
which(is.na(border_data$Value))
border_data[is.na(border_data$Value),]

#remove the row with NA value
border_data<- na.omit(border_data)
head(border_data)



[1] 71452

,Port.Name,State,Port.Code,Border,Month,Year,Measure,Value,Latitude,Longitude,Point
,<fct>,<fct>,<int>,<fct>,<chr>,<int>,<fct>,<int>,<dbl>,<dbl>,<fct>
71452,Bridgewater,Maine,127,US-Canada Border,Aug,2015,Train Passe,NA,NA,NA,


Port.Name,State,Port.Code,Border,Month,Year,Measure,Value,Latitude,Longitude,Point
<fct>,<fct>,<int>,<fct>,<chr>,<int>,<fct>,<int>,<dbl>,<dbl>,<fct>
Roma,Texas,2310,US-Mexico Border,Dec,2023,Buses,46,26.404,-99.019,POINT (-99.018981 26.403928)
Del Rio,Texas,2302,US-Mexico Border,Dec,2023,Trucks,6552,29.327,-100.928,POINT (-100.927612 29.326784)
Willow Creek,Montana,3325,US-Canada Border,Jan,2024,Pedestrians,2,49.000,-109.731,POINT (-109.731333 48.999972)
Whitlash,Montana,3321,US-Canada Border,Jan,2024,Personal Vehicles,29,48.997,-111.258,POINT (-111.257916 48.99725)
Ysleta,Texas,2401,US-Mexico Border,Jan,2024,Personal Vehicle Passengers,521714,31.673,-106.335,POINT (-106.335449846028 31.6731261376859)
Warroad,Minnesota,3423,US-Canada Border,Jan,2024,Trucks,837,48.999,-95.377,POINT (-95.376555 48.999)


In [235]:
#border_data %>% distinct(Border)
#border_data %>% distinct(Measure) 
#border_data %>% distinct(State)
#min(border_data$Year)
#max(border_data$Year)


#Select data between certain years
border_data_1924<-border_data[border_data$Year>=2019 & border_data$Year<=2024,]
head(border_data_1924)

Port.Name,State,Port.Code,Border,Month,Year,Measure,Value,Latitude,Longitude,Point
<fct>,<fct>,<int>,<fct>,<chr>,<int>,<fct>,<int>,<dbl>,<dbl>,<fct>
Roma,Texas,2310,US-Mexico Border,Dec,2023,Buses,46,26.404,-99.019,POINT (-99.018981 26.403928)
Del Rio,Texas,2302,US-Mexico Border,Dec,2023,Trucks,6552,29.327,-100.928,POINT (-100.927612 29.326784)
Willow Creek,Montana,3325,US-Canada Border,Jan,2024,Pedestrians,2,49.000,-109.731,POINT (-109.731333 48.999972)
Whitlash,Montana,3321,US-Canada Border,Jan,2024,Personal Vehicles,29,48.997,-111.258,POINT (-111.257916 48.99725)
Ysleta,Texas,2401,US-Mexico Border,Jan,2024,Personal Vehicle Passengers,521714,31.673,-106.335,POINT (-106.335449846028 31.6731261376859)
Warroad,Minnesota,3423,US-Canada Border,Jan,2024,Trucks,837,48.999,-95.377,POINT (-95.376555 48.999)


In [236]:
#First, we need to group the dataset
png(filename="entries_month.png", width = 1200, height = 800, units = "px", pointsize = 30)
pline<-border_data_1924 %>%
      filter(Measure %in% c('Pedestrians', 'Personal Vehicles', 'Personal Vehicle Passengers')) %>%
      group_by(Month, Year) %>%
      summarize(mean_val = mean(Value), .groups = 'drop') %>%
      ungroup() %>%
      mutate(
          Year = factor(Year, levels = sort(unique(Year),decreasing=FALSE))
      ) %>%  
      mutate(
       Month = factor(Month, levels = c("Jan", "Feb", "Mar", "Apr", "May", "Jun", 
                                     "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"))
      ) %>%  # convert Month to a factor with a specific order
      ggplot(aes(x = Month, y = mean_val, color = Year, group = Year)) +
      geom_line(size=5) +
      #xlab("Month") +
      #ylab("Average Number of Entries") +
      #labs(color = "Years") +
      scale_color_brewer(palette = "Paired") +
      theme_minimal() +
      theme(
        axis.title.y=element_blank(),
        axis.title.x=element_blank(),
        axis.text.y = element_text(family = "Arial",  size = 40),
        axis.text.x = element_text(family = "Arial",  size = 40),
        legend.title=element_blank(),
        legend.position = "bottom",
        legend.text = element_text(size = 30) 
      )


 pline
dev.off()



png 
  2

In [237]:
library(RColorBrewer)

png(filename="entries_year.png", width = 1200, height = 800, units = "px", pointsize = 30)


pl_entry<- border_data_1924 %>%
  filter(Measure %in% c('Pedestrians', 'Personal Vehicles', 'Personal Vehicle Passengers')) %>%
  group_by(Year,Measure) %>%
  summarize(mean_val = mean(Value,na.rm=TRUE)) %>%
  ggplot(aes(x=Year, y=mean_val, group=Measure,color=Measure)) +
  geom_line(size=5) +
  #geom_smooth(method = "lm",formula = y ~ x) +
  #xlab("Year") + ylab("Average Border Entry")+
  scale_color_brewer(palette = "Paired") +
  labs(color="Mod of Entry") +
  theme_minimal() +
  theme(
        panel.border = element_blank(),                   
        axis.line.x = element_line(color = "black", size = 1),  
        axis.line.y = element_line(color = "black", size = 1),  
        axis.title.y=element_blank(),
        axis.title.x=element_blank(),
        axis.text.y = element_text(family = "Arial",  size = 40),
        axis.text.x = element_text(family = "Arial",  size = 40),
        legend.position = "bottom",
        legend.title = element_text(size = 15),
        legend.text = element_text(size = 25) 
    )
  

pl_entry

dev.off()


png 
  2

In [238]:
entry<- border_data_1924 %>%
  filter(Measure %in% c('Pedestrians', 'Personal Vehicles', 'Personal Vehicle Passengers')) %>%
  group_by(Year) %>%
  summarize(mean_val = mean(Value,na.rm=TRUE))
entry

Year,mean_val
<int>,<dbl>
2019,100831.43
2020,54700.93
2021,61517.30
2022,86307.97
2023,96021.03
2024,99321.73


In [239]:
#Read the csv files of complaints about use of force by border officers to dataframe
incident_data_1922 = read.csv('data/use-of-force-incidents-officer-agent-fy19-fy22.csv')

incident_data_2124= read.csv('data/use-of-force-incidents-officer-agent-fy21-fy24.csv')

#Remove 2021 and 2022 data from fy19-fy22 data
incident_data_1920<-incident_data_1922%>% filter(Fiscal.Year %in% c('2019','2020'))
#Since 2024 year data is not complete yet, I will remove 2024 from the final data frame.
incident_data_2123<-incident_data_2124%>% filter(Fiscal.Year %in% c('2021','2022','2023'))
#Combine the two data frames
incident_data_1923<-rbind(incident_data_1920,incident_data_2123)
head(incident_data_1923)

Fiscal.Year,Month..abbv.,Component,Region,Area.of.Responsibility,Unique.ID,Count.of.Officers.Agents
<int>,<fct>,<fct>,<fct>,<fct>,<int>,<int>
2020,MAR,U.S. Border Patrol,Southern Border,Tucson Sector,200300057,1
2020,MAR,Office of Field Operations,Northern Border,Boston Field Office,200300072,1
2020,AUG,U.S. Border Patrol,Southern Border,El Centro Sector,200800029,1
2019,MAR,U.S. Border Patrol,Southern Border,Del Rio Sector,190300006,1
2020,DEC,U.S. Border Patrol,Southern Border,El Centro Sector,201200007,1
2020,NOV,U.S. Border Patrol,Southern Border,Laredo Sector,201100028,1


In [240]:
png(filename="incidents_month.png", width = 1200, height = 800, units = "px", pointsize = 30)
incident_data_1923%>%
                      filter(Component %in% c('Office of Field Operations'))%>%
                      group_by(Month..abbv.) %>%
                      ggplot(aes(x=Month..abbv., weight=Count.of.Officers.Agents)) +
                      geom_bar(aes(fill = ifelse(Month..abbv. == "JUL", "july", "other")),width=0.5)+                     
                      scale_fill_manual(values = c("july" = "#74c476", "other" = "darkgrey")) +                      
                      scale_x_discrete(limits=c("JAN","FEB","MAR","APR","MAY","JUN","JUL","AUG","SEP","OCT","DEC","NOV")) +
                      #xlab("Month") + ylab("Number of Officers Involved in Incidents") +
                      theme_minimal() +
                      theme(
                        panel.grid.major = element_blank(),  
                        panel.grid.minor = element_blank(),
                        axis.title.y=element_blank(),
                        axis.title.x=element_blank(),
                        axis.text.y = element_text(family = "Arial",  size = 40),
                        axis.text.x = element_text(family = "Arial",  size = 40),
                        legend.position = "none"
                      )

dev.off()

png 
  2

In [241]:
#incident_data_1923 %>% distinct(Region)

png(filename="incident_region.png", width = 1200, height = 800, units = "px", pointsize = 30)

pbar_region<-incident_data_1923%>%
                      filter(Region %in% c('Northern Border','Southern Border'))%>%
                      group_by(Region) %>%
                      ggplot(aes(x=Region)) +
                      geom_bar(stat="count",aes(fill = ifelse(Region == "Southern Border", "southernborder", "other")),width=0.4)+
                      geom_text(stat = "count", aes(label = ..count..), vjust = -0.2,size=14,color="black") +                      
                      scale_fill_manual(values = c("southernborder" = "#74c476", "other" = "darkgrey")) +                      
                      #xlab("Region") + ylab("Number of incidents") +
                      theme_minimal() +
                      theme(
                        panel.grid.major = element_blank(),  
                        panel.grid.minor = element_blank(),
                        axis.text.y=element_blank(),
                        axis.title.y=element_blank(),
                        axis.title.x = element_blank(),
                        axis.text.x = element_text(family = "Arial", size = 40),
                        legend.position = "none"
                      )
pbar_region

dev.off()

png 
  2

In [242]:
# Draw a trend graphic using incident_data_1924 dataset 
#in order to see if there is an increase in the number of use of force by border officers
library(grid)

png(filename="incident_year.png", width = 1200, height = 800, units = "px", pointsize = 30)

pl_incident<- incident_data_1923%>%
                      group_by(Fiscal.Year) %>%
                      summarize(sum_val = sum(Count.of.Officers.Agents)) %>%
                      ggplot(aes(x=Fiscal.Year, y=sum_val)) +
                      geom_line(color="#ef6548",size=10, arrow = arrow(type = "closed", length = unit(0.2, "inches")))+
                      geom_text(aes(label = Fiscal.Year), hjust=1.5,vjust = 1, size = 20, color = "black",family = "Arial") + #Years over trend line
                      theme_minimal() +

                      theme(
                    
                            panel.grid.major = element_blank(),  
                            panel.grid.minor = element_blank(),    
                            axis.line.y = element_blank(),          
                            axis.text.y = element_blank(),     
                            axis.ticks.y = element_blank(),
                            axis.title.y = element_blank(), 
                            axis.title.x = element_blank(),
                            axis.text.x = element_blank()
                      ) 

pl_incident

dev.off()


png 
  2

In [243]:

png(filename="entries_state.png", width = 1200, height = 800, units = "px", pointsize = 30)

#Barplot for port of entries by states
library(forcats)

#border_data_1924 %>% distinct(Measure)

pbar_state<-border_data_1924 %>%
          filter(Measure %in% c('Pedestrians', 'Personal Vehicles', 'Personal Vehicle Passengers')) %>%
          group_by(State) %>%
          summarize(total_value = sum(Value, na.rm = TRUE) ) %>%  # Aggregate Value by State
          filter(total_value > 20000000) %>%  # Only keep rows with total_value above 20000000
          mutate(State = fct_reorder(State, total_value), top3 = ifelse(rank(-total_value) <= 3, "Top 3", "Others")) %>%  
          ggplot(aes(x = State, y = total_value, fill = top3)) + 
          geom_bar(stat = "identity", width = 0.6) +         
          coord_flip() +
          scale_fill_manual(values = c("Top 3" = "#74c476", "Others" = "darkgrey")) + 
          geom_text(data = . %>% filter(top3 == "Top 3"), 
            aes(label = total_value), hjust = 1, size = 15,color="black") +
          theme_minimal() +
          labs(caption = "* Only for Pedestrians, Personal Vehicles, and Personal Vehicle Passengers") +

          theme(
            plot.caption = element_text(size = 20, color = "black", face = "italic"),
            panel.grid.major = element_blank(),  
            panel.grid.minor = element_blank(),
            axis.title.y = element_blank(),
            axis.text.y = element_text(family = "Arial", size = 40),
            axis.text.x = element_blank(),
            axis.ticks.x = element_blank(),
            axis.title.x = element_blank(),
            legend.position = "none"
          )

pbar_state

dev.off()


png 
  2

In [244]:
library(maps)
library(maptools)
library(ggmap)
library(sp)

png(filename="map_ports.png", width = 1200, height = 800, units = "px", pointsize = 30)

apikey <- scan("/dsa/data/all_datasets/ggmap_api_key.txt", what="character")
register_google(key = apikey)
# get the map 
us_map <- get_map(location = "United States", zoom = 4)  

# Plot the map
USmap<-ggmap(us_map, extent="device",legend = "bottomleft", darken = c(.3,"white"))

USmap +

geom_point(data=border_data_1924, aes(x = Longitude, y = Latitude, color = Value, size = Value), alpha=0.2) + 

scale_color_gradient(low = "orange", high = "red") + 

scale_alpha(range = c(0.1, 0.4), guide = FALSE)+
scale_size_continuous(range = c(3, 12)) +
guides(size = FALSE)+
theme(
legend.title=element_blank(),
legend.text= element_text(family = "Arial", size = 10)
)

dev.off()

Source : https://maps.googleapis.com/maps/api/staticmap?center=United%20States&zoom=4&size=640x640&scale=2&maptype=terrain&language=en-EN&key=xxx
Source : https://maps.googleapis.com/maps/api/geocode/json?address=United+States&key=xxx
Warning message:
“Removed 12396 rows containing missing values (geom_point).”

png 
  2